In [ ]:
#In Terminal
#pip install -U "psycopg[binary]"

#Resources: https://www.postgresql.org/docs/current/tutorial-sql-intro.html

In [1]:
import psycopg

#print('Connecting to the PostgreSQL database...')
conn = psycopg.connect(
    host="localhost",
    port='5432',
    dbname="postgres",
    user="postgres",
    password="postgres")

In [53]:
cur = conn.cursor()

In [ ]:
#list the existing tables in your current database
cur.execute("""SELECT table_name FROM information_schema.tables
       WHERE table_schema = 'public'""")
for table in cur.fetchall():
    print(table)

('users',)
('parts',)


In [54]:
# Only execute if a previous run had an error
# Rollback the failed transaction to reset the connection
conn.rollback() 
cur = conn.cursor()

In [58]:
# execute a statement
print('PostgreSQL database version:')
cur.execute('SELECT version()')

# display the PostgreSQL database server version
db_version = cur.fetchone()
print(db_version)
       
createCmd = """ CREATE TABLE instructor (
                iid CHAR(5),
                iname VARCHAR(255) NOT NULL,
                dept_name VARCHAR(255) NOT NULL,
                salary NUMERIC(8,2),
                PRIMARY KEY (iid)
                )
            """
    
cur.execute(createCmd)
conn.commit()

PostgreSQL database version:
('PostgreSQL 18.6 (Postgres.app) on aarch64-apple-darwin23.6.0, compiled by Apple clang version 15.0.0 (clang-1500.3.9.4), 64-bit',)


In [56]:
# Create the course table
createCmd = """ CREATE TABLE course(
                cid CHAR(8),
                title VARCHAR(255) NOT NULL,
                inst_name VARCHAR(255) NOT NULL,
                creadit INTEGER,
                PRIMARY KEY (cid)
                )
            """

cur.execute(createCmd)
conn.commit()

In [59]:
#insert rows into instructor table
cur.execute("insert into instructor values('1234', 'Mickey Mouse', 'APAN', 50000)"),
cur.execute("insert into instructor values('1243', 'Tinker Bell',  'STAT', 55000)"),
cur.execute("insert into instructor values('1324', 'Donald Duck',  'APAN', 40000)"),
cur.execute("insert into instructor values('1423', 'Minnie Mouse', 'APAN', 60000)"),
cur.execute("insert into instructor values('2314', 'Daisy Duck',   'APAN', 45000)"),
cur.execute("insert into instructor values('2134', 'Cheshire Cat', 'ECON', 65000)")

conn.commit()


In [57]:
#insert rows into course table
cur.execute("insert into course values('APAN5400', 'Managing Data', 'Mickey Mouse', 3)"),
cur.execute("insert into course values('APAN5450', 'Cloud Computing', 'Minnie Mouse', 3)"),
cur.execute("insert into course values('APAN5430', 'Applied Text & Natural Language Analytics', 'Donald Duck', 3)"),
cur.execute("insert into course values('APAN5800', 'Storytelling with Data', 'Daisy Duck', 3)"),
cur.execute("insert into course values('APAN5300', 'Research Design', 'Tinker Bell', 3)"),
cur.execute("insert into course values('APAN5210', 'Python for Data Analysis', 'Snow White', 3)")

conn.commit()

In [61]:
queryCmd = 'SELECT * from instructor'
cur.execute(queryCmd)    
print("The number of rows: ", cur.rowcount)

row = cur.fetchone()
while row is not None:
    print("iid:", row[0], ", instructor_name", row[1], "salary", row[3])
    row = cur.fetchone()    

The number of rows:  6
iid: 1423  , instructor_name Minnie Mouse salary 60000.00
iid: 2134  , instructor_name Cheshire Cat salary 65000.00
iid: 1234  , instructor_name Mickey Mouse salary 62500.00
iid: 1243  , instructor_name Tinker Bell salary 68750.00
iid: 1324  , instructor_name Donald Duck salary 50000.00
iid: 2314  , instructor_name Daisy Duck salary 56250.00


In [40]:
iid = 1234
cur.execute("DELETE FROM instructor WHERE iid = %s", (str(iid),))
conn.commit()

In [60]:
#Updating a record
queryCmd = 'update instructor set salary = salary * 1.25 where salary < 60000'
cur.execute(queryCmd) 

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost database=postgres) at 0x10cfd7950>

In [ ]:
#querying and retrieving records by merging two tables

query = """
    SELECT Teacher.iname, Course.title
    FROM instructor AS Teacher, course AS Course
    WHERE Teacher.salary > 50000 
      AND Teacher.iname = Course.inst_name
"""

cur.execute(query)
results = cur.fetchall()

# Loop through and print the rows
for row in results:
    print(row)

('Mickey Mouse', 'Managing Data')
('Minnie Mouse', 'Cloud Computing')
('Daisy Duck', 'Storytelling with Data')
('Tinker Bell', 'Research Design')


In [63]:
query = """
    SELECT instructor.iid, instructor.salary, course.cid, course.title
    FROM instructor
    INNER JOIN course ON instructor.iname = course.inst_name
"""

cur.execute(query)
results = cur.fetchall()

# Print the retrieved records
for row in results:
    print(row)

('1423 ', Decimal('60000.00'), 'APAN5450', 'Cloud Computing')
('1234 ', Decimal('62500.00'), 'APAN5400', 'Managing Data')
('1243 ', Decimal('68750.00'), 'APAN5300', 'Research Design')
('1324 ', Decimal('50000.00'), 'APAN5430', 'Applied Text & Natural Language Analytics')
('2314 ', Decimal('56250.00'), 'APAN5800', 'Storytelling with Data')


In [64]:
cur.execute("SELECT DISTINCT inst_name FROM course")
results = cur.fetchall()

# Print the unique instructor names
for row in results:
    print(row[0])  # row[0] extracts the name from the single-element tuple

Snow White
Tinker Bell
Mickey Mouse
Donald Duck
Minnie Mouse
Daisy Duck


In [ ]:
# Find rows where instructor name is like Mickey
query = "SELECT * FROM course WHERE inst_name LIKE '%%Mickey%%'"

cur.execute(query)
results = cur.fetchall()

for row in results:
    print(row)

('APAN5400', 'Managing Data', 'Mickey Mouse', 3)


In [66]:
query = "SELECT * FROM instructor ORDER BY iname DESC"

cur.execute(query)
results = cur.fetchall()

# Print the rows sorted in descending order
for row in results:
    print(row)

('1243 ', 'Tinker Bell', 'STAT', Decimal('68750.00'))
('1423 ', 'Minnie Mouse', 'APAN', Decimal('60000.00'))
('1234 ', 'Mickey Mouse', 'APAN', Decimal('62500.00'))
('1324 ', 'Donald Duck', 'APAN', Decimal('50000.00'))
('2314 ', 'Daisy Duck', 'APAN', Decimal('56250.00'))
('2134 ', 'Cheshire Cat', 'ECON', Decimal('65000.00'))


In [68]:
#Aggregations
cur.execute("SELECT sum(salary) FROM instructor")

# fetchone() grabs the single row result tuple: (total_salary,)
result = cur.fetchone()

# Extract the value from the tuple
total_salary = result[0] if result else 0

print(f"Total Salary for all instructors: {total_salary}")

Total Salary for all instructors: 362500.00


In [55]:
cur.execute("DROP TABLE course")
conn.commit()